# LendingClub Loan Default & Profitability Prediction
**Classification + Regression pipeline → one Word report of all metrics**

This notebook re-implements the experiments of the report *"LendingClub Loan Default and Profitability Prediction"* (Li & Han) using the code in the `zayaan-06/ML_Inspired` repository (`utils.py`, `log_reg.py`, `random_forest.py`, `perceptron.py`, `ada_boost.py`, `gaussian_nb.py`, `linear_svc.py`, `sgd_classifier.py`, `voting.py`, `pca.py`, `roc_auc_*.py`, `linear_regression.py`, `ridge.py`, `linear_svr.py`, `mlp_regressor.py`, `random_forest_regressor.py`, `random_forest_shallow_regressor*.py`, `regression_cost_eval.py`, `regression_eval.py`).

**What is identical to the repo:** feature list, missing-value rules (mean-set / zero-set / max-set ×5), percentage parsing, `earliest_cr_line` → days since epoch, one-hot encoding, 70/30 random split, `StandardScaler` fitted on train only, the regression label formula, and **every model hyper-parameter**.

**What is different (only what is necessary):**
* The repo reads a pre-cleaned `data_clean.csv`; here we read the raw Kaggle/LendingClub file `accepted_2007_to_2018Q4.csv` and keep **only loans issued 2012-2015** with a final status (`Fully Paid`, `Charged Off`, `Default`).
* `n_jobs=-1` is set on every model that supports it (affects speed only, not results).
* A few lines were ported to current pandas/scikit-learn (see the *Notes* section of the generated report).

**Outputs** (in `outputs/`): `LendingClub_ML_Metrics.docx` (all metrics), `results.json` (raw metrics checkpoint) and the PNG figures used in the report.

## 1. Setup & configuration

In [9]:
# %pip install -q pandas numpy scikit-learn matplotlib python-docx

In [10]:
import os, time, json, warnings, datetime
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ----------------------------- paths -----------------------------
CSV_PATH = os.environ.get('LC_CSV', 'accepted_2007_to_2018Q4 (1).csv')   # put the real CSV next to the notebook
OUT_DIR  = os.environ.get('LC_OUT', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------- data ------------------------------
YEAR_MIN, YEAR_MAX = 2012, 2015      # only loans issued in this range are used
FINAL_STATUSES = ['Fully Paid', 'Charged Off', 'Default']
TRAIN_FRAC  = 0.7                    # repo: np.split(..., [int(.7*len(df))])
SEED        = 42                     # makes the random split / model randomness reproducible
SAMPLE_FRAC = 0.1                  # e.g. 0.05 for a quick smoke test; None = full data

# ------------------------ what to run ----------------------------
# Set any entry to False to skip it (AdaBoost-200 and the Random Forests are the slowest).
RUN_CLASSIFIERS = {
    'Logistic Regression': True,
    'SGD Classifier': False,
    'Random Forest': True,
    'MLP (Neural Network)': True,
    'AdaBoost': False,
    'Gaussian Naive Bayes': False,
    'Linear SVC': True,
}
RUN_VOTING = False         # majority vote of AdaBoost, LogReg, MLP, RF, LinearSVC (needs all five above)
RUN_PCA    = True          # repo pca.py: components needed for 95% variance
RUN_REGRESSORS = {
    'Linear Regression': True,
    'Ridge Regression': True,
    'Linear SVR': False,
    'MLP Regressor (20,10,5,3)': True,
    'Random Forest Regressor': True,                  # n_estimators=100, min_samples_split=0.01
    'RF Regressor (max_depth=4)': True,
    'RF Regressor (max_depth=5)': True,
    'RF Regressor (max_depth=6)': True,
    'RF Regressor (max_depth=7)': True,
    'RF Regressor (max_depth=8)': True,
    'RF Regressor (max_depth=9)': True,
    'RF Regressor (max_depth=10)': True,
    # kernel_ridge.py exists in the repo, but KernelRidge builds an n x n kernel matrix
    # (~500k x 500k doubles = ~2 TB) so it cannot run on the full data. Leave False unless SAMPLE_FRAC is tiny.
    'Kernel Ridge': False,
}
STRATEGY_MODEL = 'RF Regressor (max_depth=10)'   # repo regression_eval.py uses the depth-10 forest
STRATEGY_TARGET_RETURN = 0.156                   # repo regression_eval.py threshold for the cutoff search

np.random.seed(SEED)     # sklearn models created with random_state=None draw from the global RNG

def banner(msg):
    print('\n' + '=' * 78 + f'\n{msg}\n' + '=' * 78, flush=True)

print('CSV  :', CSV_PATH)
print('Years:', YEAR_MIN, '-', YEAR_MAX)

CSV  : accepted_2007_to_2018Q4 (1).csv
Years: 2012 - 2015


## 2. Feature lists & missing-value rules (copied verbatim from the repo's `utils.py`)

In [11]:
CLS_USECOLS = ['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'emp_length', 'home_ownership',
 'annual_inc', 'verification_status', 'loan_status', 'purpose', 'zip_code', 'addr_state', 'dti',
 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths',
 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal',
 'revol_util', 'total_acc', 'out_prncp', 'out_prncp_inv', 'last_fico_range_high',
 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code',
 'application_type', 'annual_inc_joint', 'dti_joint', 'acc_now_delinq', 'tot_coll_amt',
 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'all_util', 'total_rev_hi_lim', 'inq_fi',
 'total_cu_tl', 'inq_last_12m', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc',
 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq',
 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl',
 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats',
 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq',
 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort',
 'total_bc_limit', 'total_il_high_credit_limit']

REG_USECOLS = ['loan_amnt', 'funded_amnt', 'total_pymnt', 'term', 'int_rate', 'installment', 'issue_d',
 'last_pymnt_d', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose',
 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low',
 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'out_prncp', 'out_prncp_inv',
 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med',
 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint',
 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_12m',
 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m',
 'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'acc_open_past_24mths',
 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt',
 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq',
 'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0',
 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_liens', 'tot_hi_cred_lim',
 'total_bal_ex_mort', 'total_bc_limit', 'total_il_high_credit_limit']

NAN_MAX_COLS = ['mths_since_last_delinq', 'mths_since_last_record', 'mths_since_last_major_derog',
 'mths_since_rcnt_il', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq',
 'mths_since_recent_revol_delinq', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl']

NAN_ZERO_COLS = ['tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m',
 'total_bal_il', 'open_rv_12m', 'open_rv_24m', 'inq_fi', 'inq_last_12m', 'num_accts_ever_120_pd',
 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd',
 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'mort_acc', 'percent_bc_gt_75', 'tot_hi_cred_lim',
 'total_bal_ex_mort', 'total_bc_limit', 'total_il_high_credit_limit', 'bc_open_to_buy',
 'acc_open_past_24mths', 'total_cu_tl', 'total_rev_hi_lim']

NAN_MEAN_COLS = ['dti', 'annual_inc_joint', 'dti_joint', 'il_util', 'all_util', 'bc_util', 'pct_tl_nvr_dlq',
 'avg_cur_bal', 'revol_util']

ORIG_DTYPES = {'addr_state': 'category',
 'annual_inc': 'float64',
 'application_type': 'category',
 'delinq_2yrs': 'int64',
 'dti': 'float64',
 'emp_length': 'category',
 'fico_range_high': 'int64',
 'fico_range_low': 'int64',
 'home_ownership': 'category',
 'inq_last_6mths': 'int64',
 'int_rate': 'object',
 'loan_amnt': 'int64',
 'loan_status': 'category',
 'open_acc': 'int64',
 'pub_rec': 'int64',
 'purpose': 'category',
 'revol_bal': 'int64',
 'revol_util': 'object',
 'term': 'category',
 'total_acc': 'int64',
 'verification_status': 'category',
 'zip_code': 'category'}

# Columns that are plain text / categorical in the CSV and must be read as strings.
CAT_COLS = ['term', 'emp_length', 'home_ownership', 'verification_status', 'purpose',
            'zip_code', 'addr_state', 'application_type']
# Columns the repo declares as int64 -> a NaN there means a malformed row, so such rows are dropped.
INT_REQUIRED = [c for c, t in ORIG_DTYPES.items() if t == 'int64']

RAW_COLS = sorted(set(CLS_USECOLS) | set(REG_USECOLS) | {'issue_d', 'last_pymnt_d', 'total_pymnt'})
print(len(CLS_USECOLS), 'classification columns |', len(REG_USECOLS), 'regression columns |', len(RAW_COLS), 'columns read from the CSV')

90 classification columns | 92 regression columns | 93 columns read from the CSV


## 3. Load the CSV and keep only 2012-2015 finalized loans

In [12]:
def load_raw(csv_path, year_min, year_max, chunksize=200_000):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f'{csv_path} not found. Put accepted_2007_to_2018Q4.csv next to the notebook (or set CSV_PATH).')
    if os.path.getsize(csv_path) < 1_000_000:
        raise ValueError(f'{csv_path} is only {os.path.getsize(csv_path)} bytes - it looks like a shortcut/alias, '
                         'not the real dataset. Copy the actual CSV file (~1.5 GB) instead.')
    header = pd.read_csv(csv_path, nrows=0).columns
    missing = [c for c in RAW_COLS if c not in header]
    if missing:
        raise KeyError(f'Columns missing from the CSV: {missing}')

    str_cols = {c: 'object' for c in CAT_COLS + ['int_rate', 'revol_util', 'issue_d', 'last_pymnt_d',
                                                  'earliest_cr_line', 'loan_status']}
    parts, n_read = [], 0
    for chunk in pd.read_csv(csv_path, usecols=RAW_COLS, dtype=str_cols, chunksize=chunksize, low_memory=False):
        n_read += len(chunk)
        year = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce').dt.year   # footer/junk rows -> NaT
        keep = year.between(year_min, year_max) & chunk['loan_status'].isin(FINAL_STATUSES)
        parts.append(chunk[keep])
        print(f'  read {n_read:>9,} rows, kept so far {sum(len(p) for p in parts):>9,}', end='\r', flush=True)
    raw = pd.concat(parts, ignore_index=True)
    # A footer/junk row can make a numeric column come back as text in one chunk -> force numeric dtypes.
    num_cols = [c for c in RAW_COLS if c not in str_cols]
    raw[num_cols] = raw[num_cols].apply(pd.to_numeric, errors='coerce')
    print(f'\nRows read: {n_read:,} | kept (issued {year_min}-{year_max}, final status): {len(raw):,}')
    return raw, n_read

banner('Loading data')
t0 = time.time()
RAW, N_ROWS_CSV = load_raw(CSV_PATH, YEAR_MIN, YEAR_MAX)

# Rows with NaN in the columns the repo reads as int64 would have crashed the repo's loader -> drop them.
n_before = len(RAW)
RAW = RAW.dropna(subset=INT_REQUIRED).reset_index(drop=True)
N_DROPPED_INT = n_before - len(RAW)
if SAMPLE_FRAC:
    RAW = RAW.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)
print(f'Dropped {N_DROPPED_INT:,} rows with NaN in integer-typed columns; working with {len(RAW):,} loans.  ({time.time()-t0:.0f}s)')
print(RAW['loan_status'].value_counts())


Loading data
  read 2,260,701 rows, kept so far   786,820
Rows read: 2,260,701 | kept (issued 2012-2015, final status): 786,820
Dropped 0 rows with NaN in integer-typed columns; working with 78,682 loans.  (22s)
loan_status
Fully Paid     64026
Charged Off    14655
Default            1
Name: count, dtype: int64


## 4. Preprocessing (same logic as the repo's `prepare_data` / `prepare_regression_data`)

In [13]:
from sklearn.preprocessing import StandardScaler

META = {}   # facts for the report

def clean_common(df):
    """Shared steps from utils.py: percentage parsing, NaN rules, earliest_cr_line -> days."""
    df = df.copy()
    # parse_percentage(): '13.99%' -> 0.1399  (also works if the file already stores plain numbers like 13.99)
    for c in ['revol_util', 'int_rate']:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace('%', '', regex=False), errors='coerce') / 100.0

    for col in df.columns:                                    # same loop / same order as the repo
        if col in NAN_MEAN_COLS:
            df[col] = df[col].fillna(np.nanmean(df[col].values))
        if col in NAN_ZERO_COLS:
            df[col] = df[col].fillna(0.0)
        if col in NAN_MAX_COLS:
            df[col] = df[col].fillna(np.nanmax(df[col].values) * 5)

    # repo: pd.to_timedelta(pd.to_datetime(x)).dt.days  == days since 1970-01-01
    ecl = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
    df['earliest_cr_line'] = (ecl - pd.Timestamp('1970-01-01')).dt.days
    return df

def fill_residual_nan(df, tag):
    """Safety net (not in the repo): any NaN still left would crash scikit-learn. Normally 0 cells."""
    num = df.select_dtypes('number')
    na = num.columns[num.isna().any()]
    n_cells = int(num[na].isna().sum().sum()) if len(na) else 0
    for c in na:
        m = df[c].mean()
        df[c] = df[c].fillna(0.0 if np.isnan(m) else m)
    META[f'residual_nan_cells_{tag}'] = n_cells
    if n_cells:
        print(f'[{tag}] filled {n_cells:,} residual NaN cells in {len(na)} columns with the column mean')
    return df

def split_and_scale(df, label_col, tag):
    """70/30 random split + StandardScaler fitted on the train part only (as in the repo)."""
    n = len(df)
    perm = np.random.RandomState(SEED).permutation(n)
    n_train = int(TRAIN_FRAC * n)
    tr_idx, te_idx = perm[:n_train], perm[n_train:]
    feats = [c for c in df.columns if c != label_col]
    X = df[feats].to_numpy(dtype=np.float64)
    y = df[label_col].to_numpy()
    train_x, test_x = X[tr_idx], X[te_idx]
    train_y, test_y = y[tr_idx], y[te_idx]
    del X
    scaler = StandardScaler().fit(train_x)
    train_x = scaler.transform(train_x)
    test_x = scaler.transform(test_x)
    META[f'n_features_{tag}'] = len(feats)
    META[f'n_train_{tag}'] = int(len(train_y))
    META[f'n_test_{tag}'] = int(len(test_y))
    print(f'[{tag}] features: {len(feats):,} | train: {train_x.shape} | test: {test_x.shape}')
    return train_x, train_y, test_x, test_y

def prepare_data():
    """Classification dataset  (utils.prepare_data)."""
    df = clean_common(RAW[CLS_USECOLS])
    df['loan_status'] = df['loan_status'].map({'Charged Off': 0, 'Default': 0, 'Fully Paid': 1})
    df = df.dropna(subset=['loan_status'])
    df['loan_status'] = df['loan_status'].astype(int)
    df = fill_residual_nan(df, 'classification')
    df = pd.get_dummies(df, dtype=np.uint8)                 # one-hot (uint8 as in older pandas)
    META['class_counts'] = {'Default/Charged Off (0)': int((df['loan_status'] == 0).sum()),
                            'Fully Paid (1)': int((df['loan_status'] == 1).sum())}
    META['n_loans_classification'] = int(len(df))
    return split_and_scale(df, 'loan_status', 'classification')

def prepare_regression_data():
    """Regression dataset  (utils.prepare_regression_data) - label = (total_pymnt/funded_amnt)^(365/(days+30))."""
    df = clean_common(RAW[REG_USECOLS])
    df = df.dropna(subset=['last_pymnt_d'])
    issue = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
    last = pd.to_datetime(df['last_pymnt_d'], format='%b-%Y', errors='coerce')
    df['label'] = (df['total_pymnt'] / df['funded_amnt']) ** (365.0 / ((last - issue).dt.days + 30))
    df = df.drop(['total_pymnt', 'issue_d', 'last_pymnt_d'], axis=1)
    bad = ~np.isfinite(df['label'])
    META['regression_rows_dropped_bad_label'] = int(bad.sum())
    df = df[~bad]
    df = fill_residual_nan(df, 'regression')
    df = pd.get_dummies(df, dtype=np.uint8)
    META['n_loans_regression'] = int(len(df))
    return split_and_scale(df, 'label', 'regression')

## 5. Metric helpers & results store

In [14]:
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score, roc_auc_score,
                             average_precision_score, roc_curve, precision_recall_curve, r2_score, mean_squared_error)

CLS_RESULTS, REG_RESULTS = {}, {}      # name -> metrics dict (JSON-serialisable)
CLS_PREDS, REG_PREDS = {}, {}          # name -> (train_pred, test_pred) for voting / investment strategy
CURVES = {}                            # ROC / PR curve points for the report figures
EXTRA = {}                             # PCA, strategy, ...

def save_checkpoint():
    with open(os.path.join(OUT_DIR, 'results.json'), 'w') as f:
        json.dump({'meta': META, 'classification': CLS_RESULTS, 'regression': REG_RESULTS, 'extra': EXTRA},
                  f, indent=2, default=str)

def thin(a, n=2000):
    a = np.asarray(a)
    return a if len(a) <= n else a[np.linspace(0, len(a) - 1, n).astype(int)]

def metrics_from_pred(y, pred):
    cm = confusion_matrix(y, pred, labels=[0, 1])
    rep = classification_report(y, pred, labels=[0, 1], target_names=['Default', 'Paid Off'],
                                output_dict=True, zero_division=0)
    rep.pop('accuracy', None)
    return {'confusion_matrix': cm.tolist(), 'report': rep, 'accuracy': float(accuracy_score(y, pred))}

def eval_classifier(model, X, y):
    pred = model.predict(X)
    out = metrics_from_pred(y, pred)
    score = None
    if hasattr(model, 'predict_proba'):
        score = model.predict_proba(X)[:, 1]
    elif hasattr(model, 'decision_function'):
        score = model.decision_function(X)
    out['roc_auc'] = float(roc_auc_score(y, score)) if score is not None else None
    out['avg_precision'] = float(average_precision_score(y, score)) if score is not None else None
    return out, pred, score

def eval_regressor(model, X, y):
    pred = model.predict(X)
    mse = float(mean_squared_error(y, pred))
    return {'r2': float(r2_score(y, pred)), 'mse': mse, 'rmse': float(np.sqrt(mse))}, pred

## 6. Classification: load data
Preprocessing of ~700k loans × ~1,100 one-hot features. Needs roughly 16 GB RAM for the full data (use `SAMPLE_FRAC` on smaller machines).

In [15]:
banner('Preparing CLASSIFICATION dataset')
t0 = time.time()
train_x, train_y, test_x, test_y = prepare_data()
print(f'done in {time.time()-t0:.0f}s | train class balance: {np.mean(train_y):.3f} positive')


Preparing CLASSIFICATION dataset
[classification] features: 1,041 | train: (55077, 1041) | test: (23605, 1041)
done in 1s | train class balance: 0.815 positive


## 7. Classification models (hyper-parameters exactly as in the repo)

In [16]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC

CLASSIFIERS = {
    # log_reg.py
    'Logistic Regression': lambda: LogisticRegression(class_weight='balanced', solver='lbfgs', max_iter=1000),
    # sgd_classifier.py
    'SGD Classifier':      lambda: SGDClassifier(class_weight='balanced', loss='squared_hinge', tol=1e-5, max_iter=10000),
    # random_forest.py
    'Random Forest':       lambda: RandomForestClassifier(n_estimators=200, n_jobs=-1, max_depth=30, max_features=50),
    # perceptron.py
    'MLP (Neural Network)':lambda: MLPClassifier(activation='logistic', hidden_layer_sizes=(10, 10, 5, 3)),
    # ada_boost.py
    'AdaBoost':            lambda: AdaBoostClassifier(n_estimators=200),
    # gaussian_nb.py
    'Gaussian Naive Bayes':lambda: GaussianNB(),
    # linear_svc.py
    'Linear SVC':          lambda: LinearSVC(class_weight='balanced', dual=False, penalty='l2', loss='squared_hinge'),
}

def run_classifier(name):
    banner(f'Classifier: {name}')
    model = CLASSIFIERS[name]()
    t0 = time.time()
    model.fit(train_x, train_y)
    fit_time = time.time() - t0
    tr, tr_pred, _ = eval_classifier(model, train_x, train_y)
    te, te_pred, te_score = eval_classifier(model, test_x, test_y)
    CLS_RESULTS[name] = {'params': repr(model), 'fit_time_s': fit_time, 'train': tr, 'test': te}
    CLS_PREDS[name] = (tr_pred.astype(np.int8), te_pred.astype(np.int8))
    if te_score is not None:                                   # curves for the report (test set)
        fpr, tpr, _ = roc_curve(test_y, te_score)
        p, r, _ = precision_recall_curve(test_y, te_score)
        CURVES[name] = {'fpr': thin(fpr), 'tpr': thin(tpr), 'prec': thin(p), 'rec': thin(r)}
    print(f'fit {fit_time:.0f}s | test accuracy {te["accuracy"]:.4f} | '
          f'weighted P/R/F1 = {te["report"]["weighted avg"]["precision"]:.3f}/'
          f'{te["report"]["weighted avg"]["recall"]:.3f}/{te["report"]["weighted avg"]["f1-score"]:.3f}')
    print('test confusion matrix (rows=true, cols=pred):', te['confusion_matrix'])
    save_checkpoint()
    del model

for name in CLASSIFIERS:
    if RUN_CLASSIFIERS.get(name, False):
        run_classifier(name)


Classifier: Logistic Regression
fit 1s | test accuracy 0.8688 | weighted P/R/F1 = 0.899/0.869/0.877
test confusion matrix (rows=true, cols=pred): [[3875, 571], [2527, 16632]]

Classifier: Random Forest
fit 8s | test accuracy 0.8933 | weighted P/R/F1 = 0.888/0.893/0.890
test confusion matrix (rows=true, cols=pred): [[2822, 1624], [894, 18265]]

Classifier: MLP (Neural Network)
fit 21s | test accuracy 0.8710 | weighted P/R/F1 = 0.875/0.871/0.873
test confusion matrix (rows=true, cols=pred): [[3099, 1347], [1699, 17460]]

Classifier: Linear SVC
fit 67s | test accuracy 0.8691 | weighted P/R/F1 = 0.900/0.869/0.878
test confusion matrix (rows=true, cols=pred): [[3894, 552], [2537, 16622]]


### 7b. Voting ensemble (`voting.py`), PCA (`pca.py`)
The voting rule is the repo's: sum the 0/1 predictions of AdaBoost, Logistic Regression, MLP, Random Forest and Linear SVC and predict *Paid Off* when at least 3 of 5 agree.

In [17]:
VOTERS = ['AdaBoost', 'Logistic Regression', 'MLP (Neural Network)', 'Random Forest', 'Linear SVC']
if RUN_VOTING and all(v in CLS_PREDS for v in VOTERS):
    banner('Voting ensemble (>=3 of 5)')
    tr_vote = (sum(CLS_PREDS[v][0].astype(int) for v in VOTERS) >= 3).astype(int)
    te_vote = (sum(CLS_PREDS[v][1].astype(int) for v in VOTERS) >= 3).astype(int)
    CLS_RESULTS['Voting Ensemble (5 models, >=3 votes)'] = {
        'params': 'majority vote of: ' + ', '.join(VOTERS), 'fit_time_s': None,
        'train': {**metrics_from_pred(train_y, tr_vote), 'roc_auc': None, 'avg_precision': None},
        'test':  {**metrics_from_pred(test_y, te_vote),  'roc_auc': None, 'avg_precision': None}}
    v = CLS_RESULTS['Voting Ensemble (5 models, >=3 votes)']['test']
    print('voting test accuracy', round(v['accuracy'], 4), '| confusion', v['confusion_matrix'])
    save_checkpoint()
else:
    print('Voting skipped (disabled or not all five voter models were run).')

if RUN_PCA:
    from sklearn.decomposition import PCA
    banner('PCA (95% variance, svd_solver="full")')
    t0 = time.time()
    pca = PCA(n_components=0.95, svd_solver='full').fit(train_x)
    EXTRA['pca'] = {'n_components_95pct': int(len(pca.explained_variance_ratio_)),
                    'variance_explained': float(np.sum(pca.explained_variance_ratio_)),
                    'n_input_features': int(train_x.shape[1]), 'time_s': time.time() - t0}
    print(EXTRA['pca'])
    del pca
    save_checkpoint()

# free the classification arrays before building the regression dataset
del train_x, test_x, train_y, test_y
import gc; gc.collect()

Voting skipped (disabled or not all five voter models were run).

PCA (95% variance, svd_solver="full")
{'n_components_95pct': 883, 'variance_explained': 0.9500725063975591, 'n_input_features': 1041, 'time_s': 3.537637948989868}


244

## 8. Regression: load data
Label = annualized gross return ratio `(total_pymnt / funded_amnt) ** (365 / (days_between_issue_and_last_payment + 30))` exactly as in `utils.prepare_regression_data` (the investment code later subtracts 1 to obtain the profit ratio).

In [18]:
banner('Preparing REGRESSION dataset')
t0 = time.time()
rtrain_x, rtrain_y, rtest_x, rtest_y = prepare_regression_data()
print(f'done in {time.time()-t0:.0f}s | label mean {rtrain_y.mean():.4f}, std {rtrain_y.std():.4f}')


Preparing REGRESSION dataset
[regression] features: 1,041 | train: (55039, 1041) | test: (23589, 1041)
done in 1s | label mean 1.0007, std 0.2214


## 9. Regression models (hyper-parameters exactly as in the repo)

In [19]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import LinearSVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.kernel_ridge import KernelRidge

REGRESSORS = {
    'Linear Regression':          lambda: LinearRegression(n_jobs=-1),                                   # linear_regression.py
    'Ridge Regression':           lambda: Ridge(),                                                       # ridge.py (alpha=1.0)
    'Linear SVR':                 lambda: LinearSVR(loss='squared_epsilon_insensitive', dual=False),     # linear_svr.py
    'MLP Regressor (20,10,5,3)':  lambda: MLPRegressor(hidden_layer_sizes=(20, 10, 5, 3), alpha=0.008,
                                                        learning_rate='adaptive', max_iter=1000),        # mlp_regressor.py
    'Random Forest Regressor':    lambda: RandomForestRegressor(n_estimators=100, min_samples_split=0.01, n_jobs=-1),  # random_forest_regressor.py
    'Kernel Ridge':               lambda: KernelRidge(),                                                 # kernel_ridge.py
}
for d in range(4, 11):   # random_forest_shallow_regressor*.py  (depth 4..10)
    REGRESSORS[f'RF Regressor (max_depth={d})'] = (lambda d=d: RandomForestRegressor(n_estimators=100, max_depth=d, n_jobs=-1))

def run_regressor(name):
    banner(f'Regressor: {name}')
    model = REGRESSORS[name]()
    t0 = time.time()
    model.fit(rtrain_x, rtrain_y)
    fit_time = time.time() - t0
    tr, tr_pred = eval_regressor(model, rtrain_x, rtrain_y)
    te, te_pred = eval_regressor(model, rtest_x, rtest_y)
    REG_RESULTS[name] = {'params': repr(model), 'fit_time_s': fit_time, 'train': tr, 'test': te}
    REG_PREDS[name] = (tr_pred, te_pred)
    print(f'fit {fit_time:.0f}s | R2 train {tr["r2"]:.4f} test {te["r2"]:.4f} | MSE train {tr["mse"]:.4f} test {te["mse"]:.4f}')
    save_checkpoint()
    del model

for name in REGRESSORS:
    if RUN_REGRESSORS.get(name, False):
        run_regressor(name)


Regressor: Linear Regression
fit 2s | R2 train 0.2322 test 0.2033 | MSE train 0.0376 test 0.0406

Regressor: Ridge Regression
fit 0s | R2 train 0.2322 test 0.2035 | MSE train 0.0376 test 0.0406

Regressor: MLP Regressor (20,10,5,3)
fit 12s | R2 train 0.6425 test 0.0196 | MSE train 0.0175 test 0.0500

Regressor: Random Forest Regressor
fit 69s | R2 train 0.3740 test 0.2723 | MSE train 0.0307 test 0.0371

Regressor: RF Regressor (max_depth=4)
fit 21s | R2 train 0.2768 test 0.2697 | MSE train 0.0354 test 0.0372

Regressor: RF Regressor (max_depth=5)
fit 25s | R2 train 0.2908 test 0.2730 | MSE train 0.0348 test 0.0371

Regressor: RF Regressor (max_depth=6)
fit 28s | R2 train 0.3089 test 0.2747 | MSE train 0.0339 test 0.0370

Regressor: RF Regressor (max_depth=7)
fit 31s | R2 train 0.3321 test 0.2756 | MSE train 0.0327 test 0.0369

Regressor: RF Regressor (max_depth=8)
fit 35s | R2 train 0.3614 test 0.2773 | MSE train 0.0313 test 0.0368

Regressor: RF Regressor (max_depth=9)
fit 37s | R2 t

## 10. Loan-selection strategy (`regression_eval.py`)
Rank the **training** loans by predicted return, find the cutoff where the cumulative average profit first drops below the target (0.156 in the repo), then invest $25 in every **test** loan whose prediction is ≥ that cutoff.

In [20]:
def investment_strategy(train_pred, train_y, test_pred, test_y, target=0.156, min_k=100, stake=25):
    ordered = train_y[np.argsort(-train_pred)] - 1.0              # profit ratio of loans, best prediction first
    n = len(ordered)
    ks = np.arange(min_k, n + 1)
    net_profit = np.cumsum(ordered)[min_k - 1:] / ks               # avg profit ratio when investing in the top-k
    invested_frac = ks / n
    below = np.where(net_profit < target)[0]
    if len(below) == 0:
        return None
    i = int(below[0])
    cutoff = float(-np.sort(-train_pred)[i])                       # same indexing as the repo
    sel_tr = train_pred >= cutoff
    sel_te = test_pred >= cutoff
    invested = stake * int(sel_te.sum())
    profit = float((stake * (test_y[sel_te] - 1.0)).sum())
    return {'cutoff_index': i, 'cutoff_prediction': cutoff, 'target_return': target,
            'train_pct_loans_selected': 100.0 * float(sel_tr.mean()),
            'train_avg_return_selected': float((train_y[sel_tr] - 1.0).mean()) if sel_tr.any() else None,
            'test_n_loans_selected': int(sel_te.sum()), 'test_pct_loans_selected': 100.0 * float(sel_te.mean()),
            'test_invested_usd': float(invested), 'test_profit_usd': profit,
            'test_return_on_invested': (profit / invested) if invested else None,
            'curve_x_pct': thin(invested_frac * 100), 'curve_y_pct': thin(net_profit * 100)}

def curve_on_test(test_pred, test_y, min_k=100):
    ordered = test_y[np.argsort(-test_pred)] - 1.0
    ks = np.arange(min_k, len(ordered) + 1)
    return thin(ks / len(ordered) * 100), thin(np.cumsum(ordered)[min_k - 1:] / ks * 100)

if STRATEGY_MODEL in REG_PREDS:
    banner(f'Loan selection strategy using: {STRATEGY_MODEL}')
    tr_pred, te_pred = REG_PREDS[STRATEGY_MODEL]
    strat = investment_strategy(tr_pred, rtrain_y, te_pred, rtest_y, target=STRATEGY_TARGET_RETURN)
    if strat is None:
        print(f'The training-set average return never drops below {STRATEGY_TARGET_RETURN:.3f}; no cutoff found.')
        EXTRA['strategy'] = None
    else:
        cx, cy = curve_on_test(te_pred, rtest_y)
        CURVES['strategy_train'] = {'x': strat.pop('curve_x_pct'), 'y': strat.pop('curve_y_pct')}
        CURVES['strategy_test'] = {'x': cx, 'y': cy}
        strat['model'] = STRATEGY_MODEL
        EXTRA['strategy'] = strat
        for k, v in strat.items():
            print(f'{k:>28}: {v}')
else:
    print(f'Strategy skipped: "{STRATEGY_MODEL}" was not run.')
save_checkpoint()


Loan selection strategy using: RF Regressor (max_depth=10)
                cutoff_index: 1033
           cutoff_prediction: 1.131495146077092
               target_return: 0.156
    train_pct_loans_selected: 1.8786678537037373
   train_avg_return_selected: 0.15926212243104773
       test_n_loans_selected: 404
     test_pct_loans_selected: 1.7126626817584467
           test_invested_usd: 10100.0
             test_profit_usd: 1457.6586222519138
     test_return_on_invested: 0.14432263586652613
                       model: RF Regressor (max_depth=10)


## 11. Figures for the report

In [21]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

FIGS = {}   # title -> path

if any(n in CURVES for n in ['Logistic Regression', 'Random Forest', 'Linear SVC', 'SGD Classifier', 'AdaBoost']):
    fig, ax = plt.subplots(figsize=(6, 5))
    for n, c in CURVES.items():
        if 'fpr' in c:
            ax.plot(c['fpr'], c['tpr'], label=f"{n} (AUC={CLS_RESULTS[n]['test']['roc_auc']:.3f})")
    ax.plot([0, 1], [0, 1], 'r--')
    ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC curves (test set)', xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc='lower right', fontsize=8)
    p = os.path.join(OUT_DIR, 'roc_test.png'); fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
    FIGS['ROC curves (test set)'] = p

if 'Linear SVC' in CURVES:
    c = CURVES['Linear SVC']
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.step(c['rec'], c['prec'], where='post', color='b', alpha=0.6)
    ax.fill_between(c['rec'], c['prec'], alpha=0.2, color='b', step='post')
    ax.set(xlabel='Recall', ylabel='Precision', ylim=(0, 1.05), xlim=(0, 1),
           title=f"Linear SVC precision-recall (test) AP={CLS_RESULTS['Linear SVC']['test']['avg_precision']:.3f}")
    p = os.path.join(OUT_DIR, 'svc_pr_test.png'); fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
    FIGS['Linear SVC precision-recall curve (test set)'] = p

for key, title in [('strategy_train', 'Investment return trend - training set'), ('strategy_test', 'Investment return trend - test set')]:
    if key in CURVES:
        with plt.style.context('Solarize_Light2'):
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.plot(CURVES[key]['x'], CURVES[key]['y'])
            ax.set(title=title, xlabel='percentage of loans invested', ylabel='annualized return rate')
            fmt = mtick.FormatStrFormatter('%.0f%%')
            ax.xaxis.set_major_formatter(fmt); ax.yaxis.set_major_formatter(fmt)
            p = os.path.join(OUT_DIR, f'{key}.png'); fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
        FIGS[title] = p
print(list(FIGS))

['ROC curves (test set)', 'Linear SVC precision-recall curve (test set)', 'Investment return trend - training set', 'Investment return trend - test set']


## 12. Build the Word report with all metrics

In [22]:
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

def f4(v):  return '-' if v is None else f'{v:.4f}'
def f3(v):  return '-' if v is None else f'{v:.3f}'

def shade(cell, hex_fill):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement('w:shd'); shd.set(qn('w:val'), 'clear'); shd.set(qn('w:color'), 'auto'); shd.set(qn('w:fill'), hex_fill)
    tcPr.append(shd)

def add_table(doc, header, rows, font_pt=8.5, col_widths=None, first_col_bold=True, center=True):
    t = doc.add_table(rows=1, cols=len(header))
    t.style = 'Table Grid'
    t.autofit = False
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for i, h in enumerate(header):
        c = t.rows[0].cells[i]; c.text = ''
        r = c.paragraphs[0].add_run(str(h)); r.bold = True; r.font.size = Pt(font_pt); r.font.color.rgb = RGBColor(255, 255, 255)
        shade(c, '1F3864')
    for ri, row in enumerate(rows):
        cells = t.add_row().cells
        for i, v in enumerate(row):
            cells[i].text = ''
            run = cells[i].paragraphs[0].add_run(str(v)); run.font.size = Pt(font_pt)
            if i == 0 and first_col_bold: run.bold = True
            if i > 0 and center: cells[i].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
            if ri % 2 == 1: shade(cells[i], 'EEF2F8')
    if col_widths:
        for i, w in enumerate(col_widths):
            t.columns[i].width = Inches(w)
        for row in t.rows:
            for i, w in enumerate(col_widths):
                row.cells[i].width = Inches(w)
    doc.add_paragraph()
    return t

def add_cm_table(doc, cm, title):
    p = doc.add_paragraph(); r = p.add_run(title); r.bold = True; r.font.size = Pt(9)
    add_table(doc, ['', 'Predicted Default', 'Predicted Paid Off'],
              [['True Default', f'{cm[0][0]:,}', f'{cm[0][1]:,}'], ['True Paid Off', f'{cm[1][0]:,}', f'{cm[1][1]:,}']],
              col_widths=[1.5, 1.6, 1.6])

def add_report_table(doc, res, title):
    p = doc.add_paragraph(); r = p.add_run(title); r.bold = True; r.font.size = Pt(9)
    rep = res['report']; rows = []
    for k in ['Default', 'Paid Off', 'macro avg', 'weighted avg']:
        d = rep[k]; rows.append([k, f3(d['precision']), f3(d['recall']), f3(d['f1-score']), f"{int(d['support']):,}"])
    rows.append(['accuracy', '', '', f4(res['accuracy']), f"{int(rep['weighted avg']['support']):,}"])
    add_table(doc, ['class', 'precision', 'recall', 'f1-score', 'support'], rows, col_widths=[1.5, 1.1, 1.1, 1.1, 1.1])

def h(doc, text, lvl): return doc.add_heading(text, level=lvl)
def para(doc, text, bold=False, italic=False, size=10):
    p = doc.add_paragraph(); r = p.add_run(text); r.bold = bold; r.italic = italic; r.font.size = Pt(size); return p
def bullet(doc, text):
    p = doc.add_paragraph(style='List Bullet'); r = p.add_run(text); r.font.size = Pt(10)

def build_report(path):
    doc = Document()
    sec = doc.sections[0]
    sec.orientation = WD_ORIENT.LANDSCAPE
    sec.page_width, sec.page_height = Inches(11), Inches(8.5)
    for m in ('left_margin', 'right_margin'): setattr(sec, m, Inches(0.7))
    sec.top_margin = sec.bottom_margin = Inches(0.7)
    doc.styles['Normal'].font.name = 'Calibri'; doc.styles['Normal'].font.size = Pt(10)

    t = doc.add_heading('LendingClub Loan Default & Profitability Prediction', 0)
    para(doc, 'Classification and regression metrics for all techniques', italic=True, size=12)
    para(doc, f'Generated {datetime.datetime.now():%Y-%m-%d %H:%M} | Loans issued {YEAR_MIN}-{YEAR_MAX} | random seed {SEED}'
              + (f' | SAMPLE_FRAC={SAMPLE_FRAC}' if SAMPLE_FRAC else ''), size=9)

    # ---------------- 1. dataset ----------------
    h(doc, '1. Dataset and preprocessing', 1)
    cc = META.get('class_counts', {})
    tot = sum(cc.values()) or 1
    rows = [
        ['Source file', os.path.basename(CSV_PATH)],
        ['Rows in CSV', f'{N_ROWS_CSV:,}'],
        ['Filter', f'issue year {YEAR_MIN}-{YEAR_MAX}; loan_status in {", ".join(FINAL_STATUSES)}'],
        ['Loans kept (after dropping NaN in integer columns)', f'{len(RAW):,}'],
        ['Classification loans', f"{META.get('n_loans_classification', 0):,}"],
        ['Class balance (0 = Default/Charged Off, 1 = Fully Paid)',
         ' | '.join(f'{k}: {v:,} ({100*v/tot:.1f}%)' for k, v in cc.items())],
        ['Classification features (after one-hot)', f"{META.get('n_features_classification', '-'):,}" if META.get('n_features_classification') else '-'],
        ['Classification train / test', f"{META.get('n_train_classification', 0):,} / {META.get('n_test_classification', 0):,}"],
        ['Regression loans (have last_pymnt_d)', f"{META.get('n_loans_regression', 0):,}"],
        ['Regression features (after one-hot)', f"{META.get('n_features_regression', '-'):,}" if META.get('n_features_regression') else '-'],
        ['Regression train / test', f"{META.get('n_train_regression', 0):,} / {META.get('n_test_regression', 0):,}"],
        ['Split', f'random {int(TRAIN_FRAC*100)}/{100-int(TRAIN_FRAC*100)}'],
        ['Scaling', 'StandardScaler fitted on the training split only'],
        ['Missing values', 'mean-set / zero-set / max-set (x5) lists exactly as in the repo utils.py'],
    ]
    add_table(doc, ['Item', 'Value'], rows, font_pt=9, col_widths=[3.6, 5.8], center=False)
    if 'pca' in EXTRA:
        p = EXTRA['pca']
        para(doc, f"PCA (95% variance, svd_solver='full'): {p['n_components_95pct']:,} of {p['n_input_features']:,} features are needed "
                  f"({p['variance_explained']*100:.2f}% variance retained) - i.e. the feature count cannot be reduced much.")

    # ---------------- 2. classification ----------------
    h(doc, '2. Classification results', 1)
    para(doc, 'Positive class = Fully Paid (1); negative class = Default / Charged Off (0). Confusion matrices: rows = true label, columns = predicted label.')
    names = list(CLS_RESULTS)
    for split, title in [('test', '2.1 Summary - test set'), ('train', '2.2 Summary - training set')]:
        h(doc, title, 2)
        rows = []
        for n in names:
            r = CLS_RESULTS[n][split]; rep = r['report']
            rows.append([n, f4(r['accuracy']),
                         f3(rep['Default']['precision']), f3(rep['Default']['recall']), f3(rep['Default']['f1-score']),
                         f3(rep['Paid Off']['precision']), f3(rep['Paid Off']['recall']), f3(rep['Paid Off']['f1-score']),
                         f3(rep['weighted avg']['precision']), f3(rep['weighted avg']['recall']), f3(rep['weighted avg']['f1-score']),
                         f3(r.get('roc_auc')),
                         '-' if CLS_RESULTS[n]['fit_time_s'] is None else f"{CLS_RESULTS[n]['fit_time_s']:.0f}"])
        add_table(doc, ['Model', 'Accuracy', 'Default P', 'Default R', 'Default F1', 'Paid P', 'Paid R', 'Paid F1',
                        'Wtd P', 'Wtd R', 'Wtd F1', 'ROC AUC', 'Fit time (s)'], rows, font_pt=8,
                  col_widths=[2.0] + [0.65] * 12)
    h(doc, '2.3 Per-model details', 2)
    for n in names:
        res = CLS_RESULTS[n]
        h(doc, n, 3)
        para(doc, 'Configuration: ' + res['params'], size=8.5)
        if res['test'].get('avg_precision') is not None:
            para(doc, f"Test ROC AUC = {f4(res['test']['roc_auc'])} | Test average precision = {f4(res['test']['avg_precision'])} | "
                      f"Train ROC AUC = {f4(res['train']['roc_auc'])}", size=9)
        add_cm_table(doc, res['train']['confusion_matrix'], 'Training set - confusion matrix')
        add_report_table(doc, res['train'], 'Training set - classification report')
        add_cm_table(doc, res['test']['confusion_matrix'], 'Test set - confusion matrix')
        add_report_table(doc, res['test'], 'Test set - classification report')
    for title in ['ROC curves (test set)', 'Linear SVC precision-recall curve (test set)']:
        if title in FIGS:
            h(doc, title, 3); doc.add_picture(FIGS[title], width=Inches(4.8))

    # ---------------- 3. regression ----------------
    h(doc, '3. Regression results', 1)
    para(doc, 'Target = annualized return ratio (total_pymnt / funded_amnt) ** (365 / (days + 30)). Metrics: R-squared and mean squared error (RMSE = sqrt(MSE)).')
    rows = []
    for n, r in REG_RESULTS.items():
        rows.append([n, f4(r['train']['r2']), f4(r['test']['r2']), f4(r['train']['mse']), f4(r['test']['mse']),
                     f4(r['train']['rmse']), f4(r['test']['rmse']), f"{r['fit_time_s']:.0f}"])
    add_table(doc, ['Model', 'Train R2', 'Test R2', 'Train MSE', 'Test MSE', 'Train RMSE', 'Test RMSE', 'Fit time (s)'], rows,
              font_pt=8.5, col_widths=[2.6, 0.95, 0.95, 0.95, 0.95, 0.95, 0.95, 0.95])
    h(doc, '3.1 Model configurations', 2)
    for n, r in REG_RESULTS.items():
        p = doc.add_paragraph(); a = p.add_run(n + ': '); a.bold = True; a.font.size = Pt(9)
        b = p.add_run(r['params']); b.font.size = Pt(8.5)

    # ---------------- 4. strategy ----------------
    h(doc, '4. Loan selection strategy', 1)
    s = EXTRA.get('strategy')
    if s:
        para(doc, f"Model: {s['model']}. Training loans are ranked by predicted return; the cutoff is the prediction at the point where the cumulative "
                  f"average return first falls below {s['target_return']:.3f}. $25 is then invested in every test loan whose prediction >= cutoff.")
        rows = [['Cutoff (predicted return ratio)', f4(s['cutoff_prediction'])],
                ['Training: % of loans selected', f"{s['train_pct_loans_selected']:.2f}%"],
                ['Training: average return of selected loans', '-' if s['train_avg_return_selected'] is None else f"{s['train_avg_return_selected']*100:.2f}%"],
                ['Test: loans selected', f"{s['test_n_loans_selected']:,} ({s['test_pct_loans_selected']:.2f}% of test loans)"],
                ['Test: amount invested', f"${s['test_invested_usd']:,.0f}"],
                ['Test: profit', f"${s['test_profit_usd']:,.0f}"],
                ['Test: return on invested amount', '-' if s['test_return_on_invested'] is None else f"{s['test_return_on_invested']*100:.2f}%"]]
        add_table(doc, ['Metric', 'Value'], rows, font_pt=9, col_widths=[4.0, 4.0], center=False)
        for title in ['Investment return trend - training set', 'Investment return trend - test set']:
            if title in FIGS: doc.add_picture(FIGS[title], width=Inches(5.2))
    else:
        para(doc, 'Strategy was not evaluated (model not run or no cutoff found).')

    # ---------------- 5. notes ----------------
    h(doc, '5. Notes and deviations', 1)
    for t_ in [
        'Hyper-parameters, features and preprocessing follow the ML_Inspired repository (utils.py and the model scripts). Where the repository and the report differ, the repository was used: Random Forest classifier max_depth=30 (report: unrestricted), MLP regressor alpha=0.008 with learning_rate="adaptive" (report: 0.0001).',
        'n_jobs=-1 was added to models that support it; this only affects speed.',
        'The repo reads a cleaned file (data_clean.csv); this notebook builds the equivalent from the raw CSV by keeping loans issued 2012-2015 with status Fully Paid / Charged Off / Default.',
        'Ported for current libraries: get_dummies(dtype=uint8); earliest_cr_line converted with (date - 1970-01-01).days (same value as the old to_timedelta trick); AdaBoost uses the current scikit-learn default algorithm (SAMME).',
        f"Rows with NaN in integer-typed columns were dropped ({N_DROPPED_INT:,}); residual NaN cells (columns not covered by the repo's fill lists) were filled with column means: classification {META.get('residual_nan_cells_classification', 0):,}, regression {META.get('residual_nan_cells_regression', 0):,}.",
        'Kernel Ridge (kernel_ridge.py) and RFE logistic regression (rfe_log_reg.py, 900 features with step=1) are not run: the first needs an n x n kernel matrix, the second hundreds of full model fits.',
        'The loan-selection cutoff reproduces the repo indexing (the position found in the cumulative-return array is used as an index into the sorted predictions).',
    ]:
        bullet(doc, t_)
    doc.save(path)
    return path

REPORT_PATH = os.path.join(OUT_DIR, 'LendingClub_ML_Metrics.docx')
build_report(REPORT_PATH)
print('Saved:', os.path.abspath(REPORT_PATH))

Saved: /Users/zayaanahmed/Documents/outputs/LendingClub_ML_Metrics.docx


## Done
All metrics are in `outputs/LendingClub_ML_Metrics.docx` (and the raw numbers in `outputs/results.json`).
If a long run is interrupted, `results.json` holds everything finished so far; re-run only the cells you need after toggling the `RUN_*` flags in the configuration cell.